In [0]:
%sh
pip install optuna

## Data preparation
1. Load the cleaned spark dataframe 
2. Create mappings for user + products and generate "ratings" for training dataset
3. Use training mappings to process test dataset

In [0]:
from pyspark.sql.functions import col

catalog = 'kfcus_digital_model'
schema = 'recommender_test'

sdf_cleaned_mapped = spark.read.table(f'{catalog}.{schema}.cleaned_mapped_dataset')

## Split the data into training and test sets
train, test = sdf_cleaned_mapped.randomSplit([0.8, 0.2], seed=42)

# display(sdf_cleaned)

In [0]:
from pyspark.sql.functions import explode, col, count, sum as spark_sum, row_number
from pyspark.sql.window import Window

def preprocess_pipeline(train_dataset): 
  """
  Preprocess the training dataset for ALS (Alternating Least Squares) model.

  This function performs the following steps:
  1. Converts 'mpid' column to double type.
  2. Explodes the 'order_product_list' column to have one product per row.
  3. Renames columns: 'mpid' to 'user' and 'product' to 'item'.
  4. Creates mappings for 'item' (string) and 'user' (double) for ALS.
  5. Joins the mappings with the ALS input DataFrame.
  6. Creates a feature "rating" by calculating the proportion of orders that include a particular item.

  Args:
  train_dataset (DataFrame): The input training dataset.

  Returns:
  tuple: A tuple containing:
    - als_train_dataset (DataFrame): The preprocessed dataset ready for ALS model.
    - item_mapping (DataFrame): The mapping of items to item IDs.
    - user_mapping (DataFrame): The mapping of users to user IDs.
  """
  ## Prepare dataset for ALS
  # 1. convert mpid to a double type
  train_dataset = train_dataset.withColumn('mpid', col('mpid').cast('double'))
  # 2. explode the order_product_list to have one product per row
  sdf_exploded = train_dataset.withColumn('product', explode('order_product_list'))
  # 3. rename columns 'mpid' --> user and 'product' --> 'item'. 
  als_input = sdf_exploded.select(col('mpid').alias('user'), col('product').alias('item'))
  # 4. Create mappings for item (string) and user (double) for ALS
  item_mapping = als_input.select('item').distinct().withColumn('item_id', row_number().over(Window.orderBy('item')))
  user_mapping = als_input.select('user').distinct().withColumn('user_id', row_number().over(Window.orderBy('user')))
  # 5. Join the mappings with the ALS input DataFrame
  als_input_mapped = als_input.join(item_mapping, on='item').join(user_mapping, on='user').select('user_id', 'item_id', 'item')

  ## Create a feature "rating". We'll calculate the roportion of orders that include a particular item (i.e. item frequency in orders)
  # Count the total number of times a particular item was ordered by a user
  als_input_grouped = als_input_mapped.groupBy('user_id', 'item_id').agg(count('*').alias('historical_ordered_amount'))

  # Calculate the total number of orders for each user
  user_total_orders = als_input_grouped.groupBy('user_id').agg(spark_sum('historical_ordered_amount').alias('total_orders'))
  als_input_with_totals = als_input_grouped.join(user_total_orders, on='user_id')

  # Calculate the proportion of orders with that item
  als_train_dataset = als_input_with_totals.withColumn('proportion_of_orders', col('historical_ordered_amount') / col('total_orders'))

  return als_train_dataset, item_mapping, user_mapping

In [0]:
# run the preprocessing pipeline for our training dataset
als_train_dataset, item_mapping, user_mapping = preprocess_pipeline(train)

In [0]:
from pyspark.sql.functions import expr, size, col

# Load test dataset
test.repartition(sc.defaultParallelism*10)

# Remove rows with only one item in order_product_list
test_filtered = test.filter(size(col("order_product_list")) > 1)
# Create "cart" and "added" columns. Cart is all items with the last item in the list removed. Added is the last item in the list.
test_transformed = test_filtered.withColumn("cart", expr("slice(order_product_list, 1, size(order_product_list) - 1)")) \
                                .withColumn("added", expr("order_product_list[size(order_product_list) - 1]"))

# Link 'mpid' in test_transformed with 'user_id' in user_mapping
test_transformed = test_transformed.withColumn('mpid', col('mpid').cast('double'))
test_linked = test_transformed.join(user_mapping, test_transformed.mpid == user_mapping.user, "inner")

# Link 'added' item to 'item_id' using item_mapping
test_linked = test_linked.join(item_mapping, test_linked.added == item_mapping.item, "inner") \
                         .withColumnRenamed("item_id", "added_item_id")

display(test_linked)

test_set_count = test.count()
print(f'{test_set_count - test_linked.count()} rows out of {test_set_count} remain in test after filtering out rows with only one item')

## Construct Collaborative Filter model using ALS
Our whole training pipeline will encapsulate the below:

1. Test building one small (i.e. low rank) model to ensure our preprocessing function works
2. Evaluate our low rank model 
3. Hyperparameter tune using Optuna to find the best `rank` and `maxIter`
4. Use best hyperparameters to train final model on full dataset

In [0]:
from pyspark.ml.recommendation import ALS

def train_als(train_dataset, rank, maxIter):
  # define the als model
  als = ALS(
    rank=rank,
    maxIter=maxIter,  
    userCol='user_id', 
    itemCol='item_id', 
    ratingCol='proportion_of_orders',
    implicitPrefs=True
    )
  
  # train the model
  model = als.fit(train_dataset)
  return model

model = train_als(train_dataset=als_train_dataset,
                  rank=10,
                  maxIter=5)


### Evaluate model on test dataset
Our evaluation metric will be "Hit @ k" (What proportion of time will the user in the test dataset add the 'k' recommended items)

In [0]:
from pyspark.sql.functions import array_contains, avg

# Hit@k measures what proportion of the time is the last item that's added to the cart a top k recommended item. 
# For example, Hit@5 = 0.25 means that 25% of the time the last item to be added to the cart is in the top 5 recommendations

# number of recommendations
k = 5

# generate recommendations with our model 
test_recommendations = model.recommendForUserSubset(test_linked.select('user_id').distinct(), k)
test_recommendations = test_recommendations.join(test_linked, "user_id", "inner") \
                                           .select("mpid", "user_id", "cart", "added", "added_item_id", "recommendations") \
                                           .withColumn("hit@k", array_contains(col("recommendations.item_id"), col("added_item_id")).cast("int"))
# calcaluate average hit@k across our whole test set
average_hit_k = test_recommendations.agg(avg(col("hit@k")).alias("average_hit@k"))

display(average_hit_k)

### Hyperparameter tuning to find the best model
ALS has two knobs for us to turn to maximize its performance on the test set:
* `rank` - the size of the lower level product + user matrices
* `numIter` - the number of times we alternate between holding + optimizing the two matrices until a stable solution is found

In [0]:
from pyspark.ml.recommendation import ALS
import optuna
import mlflow
from optuna.integration.mlflow import MLflowCallback
from functools import partial
from pyspark.sql.functions import array_contains, avg


def objective(trial, parent_run_id, k=5): 
  with mlflow.start_run(parent_run_id=parent_run_id,
                        nested=True):
    
    params = {'rank': trial.suggest_int('rank', 1, 100), 
              'maxIter': trial.suggest_int('maxIter', 1, 10)}
    
    model = train_als(train_dataset=als_train_dataset,
                      rank = params['rank'],
                      maxIter= params['maxIter'])

    # evaluate the model
    test_recommendations = model.recommendForUserSubset(test_linked.select('user_id').distinct(), k)
    test_recommendations = test_recommendations.join(test_linked, "user_id", "inner") \
                                              .select("mpid", "user_id", "cart", "added", "added_item_id", "recommendations") \
                                              .withColumn("hit@k", array_contains(col("recommendations.item_id"), col("added_item_id")).cast("int"))

    average_hit_k = test_recommendations.agg(avg(col("hit@k")).alias("average_hit@k"))

    # mlflow.spark.log_model(model, "ALS")
    mlflow.log_params(params)
    mlflow.log_metric("average_hit@k", average_hit_k.first()['average_hit@k'])

    return average_hit_k.first()['average_hit@k']



In [0]:
import os

mlflow.set_experiment('/Shared/Recommender Model/ALS_recommender_model')
n_trials=20


with mlflow.start_run(run_name='ALS_hp_tuning_test') as parent_run:
  # using partial to pass into objective other parameters for child logging with mlflow
  objective = partial(objective, 
                      parent_run_id=parent_run.info.run_id, 
                      k=5)

  # create study to maximize the metric and run n_trials
  study = optuna.create_study(direction='maximize')
  study.optimize(objective, 
                n_trials=n_trials)
    
  trial = study.best_trial
  print(f"Best trial accuracy: {trial.value}")
  print("Best trial params: ")
  for key, value in trial.params.items():
      print(f"    {key}: {value}")
    
  # After hyperparameter tuning, we can train with all the data to ensure maximum performance when we dump the model into production
  with mlflow.start_run(run_name='final_als_model_250321', 
                        nested=True,
                        parent_run_id=parent_run.info.run_id):
    als_full_dataset, item_mapping, user_mapping = preprocess_pipeline(sdf_cleaned_mapped)
    als_full_dataset.repartition(numPartitions=sc.defaultParallelism*100)
    final_model = train_als(train_dataset=als_full_dataset,
                            rank=trial.params['rank'],
                            maxIter=trial.params['maxIter'])
    # write tables to a parquet so that we can log them as an artifact to mlflow
    user_mapping.write.parquet(os.getcwd()  + "/user_mapping_ALS.parquet", mode="overwrite")
    item_mapping.write.parquet(os.getcwd()  + "/item_mapping_ALS.parquet", mode="overwrite")
    # files are dumped into DBFS so we'll get it from there and log the model
    mlflow.log_artifact('/dbfs' + os.getcwd() + "/user_mapping_ALS.parquet", 'user_mapping')
    mlflow.log_artifact('/dbfs' + os.getcwd() + "/item_mapping_ALS.parquet", 'item_mapping')
    mlflow.spark.log_model(final_model, artifact_path='model')
    mlflow.log_params({'rank': trial.params['rank'], 
                       'maxIter': trial.params['maxIter']})

## Generate recommendations for all users

In [0]:
# Write the recommendations using the final model, item map, and user map to a delta table so we can serve with the online table.
k = 20 
recommendations = final_model.recommendForAllUsers(k)
recommendations.write.format('delta').mode("overwrite").saveAsTable(f"{catalog}.{schema}.als_recommendations")
user_mapping.write.format('delta').mode("overwrite").saveAsTable(f"{catalog}.{schema}.als_user_mapping")
item_mapping.write.format('delta').mode("overwrite").saveAsTable(f"{catalog}.{schema}.als_item_mapping")

## IGNORE BELOW: 
Inference function with pyfunc but inference time is too long.

In [0]:
import pandas as pd

# load in model
model_uri = 'runs:/a86c756c372848fd851087db5f8f9f20/model'
model = mlflow.spark.load_model(model_uri)
als = model.stages[0]

# load in mappings
# user_mapping = spark.read.parquet(
#     'dbfs:/Workspace/Shared/Recommender Model/user_mapping_ALS.parquet',
#     header=True,
#     inferSchema=True
# )

user_mapping = pd.read_parquet('/dbfs/Workspace/Shared/Recommender Model/user_mapping_ALS.parquet')
item_mapping = pd.read_parquet('/dbfs/Workspace/Shared/Recommender Model/item_mapping_ALS.parquet')


In [0]:
from pyspark.sql.functions import explode, col

def recommender(user, cart): 
    # Check if user is in our user mapping
    try:
        user_id = user_mapping.filter(user_mapping['user'] == user).select('user_id').collect()[0][0]
        user_subset = spark.createDataFrame([(user_id,)], ["user_id"])
    except:
        # if not, you can return None
        print('no user detected')
        user = None
    
    # NO CART // NO USER
    # recommend most popular products
    if not user and not cart:
        recommendations = ['3 pc. Chicken Box - 2 Breasts', '1 Wing', 'A La Carte Tender']
        return recommendations
    

    if user: 
        # Generate top 20 recommendations for user
        recommendations = als.recommendForUserSubset(user_subset, 20)
        recommendations = recommendations.withColumn("recommendations", 
                                                    explode("recommendations")).select("user_id", 
                                                                                        col("recommendations.item_id"), 
                                                                                        col("recommendations.rating")).toPandas()
        # YES CART // YES USER
        if cart:    
            print('cart detected, removing recommendations that are already in cart')
            # Convert current cart to the mapped ids using item_mapping
            cart_mapping = item_mapping[item_mapping['item'].isin(cart)]
            # Remove recommendations that are already in cart
            recommendations = recommendations[~recommendations['item_id'].isin(cart_mapping['item_id'])]
        
        # NO CART // YES USER
        else:
            print('no cart detected so no filtering needed')
            
        # convert recommendations to string
        recommendations = recommendations.merge(item_mapping, on='item_id').sort_values('rating', ascending=False)

    return recommendations.head(9)


In [0]:

user = -8.972000940743769e+18
# user = None
current_cart = ['biscuit', 'secret-recipe-fries']
# current_cart = None

recommendations = recommender(user, current_cart)
recommendations

In [0]:

current_cart = ['3 pc. Chicken Box - 2 Breasts', '1 Wing', 'A La Carte Tender']

item_mapping = spark.read.parquet(
    'dbfs:/Workspace/Shared/Recommender Model/item_mapping_ALS.parquet',
    header=True,
    inferSchema=True
)

# Convert current cart to the mapped ids using item_mapping
current_cart_ids = item_mapping.filter(item_mapping['item'].isin(current_cart)).select('item_id')

# Generate recommendations using als.recommendForItemSubset
recommendations = als.recommendForItemSubset(current_cart_ids, 5)

display(recommendations)